# Knee MRI — Kaggle run (single notebook)

One notebook, one running kernel: clone the repo, then run preprocess -> train -> evaluate -> predict by **importing** the same tested functions `scripts/*.py` expose (`preprocess_split`, `run_training`, `evaluate_and_save`, `run_prediction`) instead of shelling out to `!python scripts/*.py` as separate processes.

Why this instead of separate `!python` calls: those only hand off state through a checkpoint *file path* you have to type correctly for every step (config's `run_name` -> `checkpoint_root` -> `best.pt`) -- easy to get out of sync (e.g. running `train.py` against one config's `run_name` and `evaluate.py` against another's). Here the checkpoint path comes directly from the training call's return value, so there's nothing to get wrong.

**This does not change disk usage** -- the series-tensor cache under `/kaggle/working/cache` is identical either way. See `README.md`'s "Running on Kaggle" section for the confirmed 19.5 GB quota and why every config here stays under a `data.max_studies` cap rather than running on the literal full dataset.

In [ ]:
import os

REPO_DIR = '/kaggle/working/RSNA-Knee-Abnormality-Detection'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/rebhimohamedamine/RSNA-Knee-Abnormality-Detection.git {REPO_DIR}
%cd {REPO_DIR}
!git pull
!pip install -r requirements.txt -q

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path(REPO_DIR)
sys.path.insert(0, str(REPO_ROOT))          # so `from src....` resolves
sys.path.insert(0, str(REPO_ROOT / 'scripts'))  # scripts/ is deliberately not a package -- import its files as top-level modules

from src.utils.config import load_config

import preprocess as preprocess_script
import train as train_script
import evaluate as evaluate_script
import predict as predict_script

device = train_script.select_device('cuda')
print('Using device:', device)

## Step 1 — Smoke test on real data

`configs/kaggle_smoke.yaml`: 150 real studies, M1 baseline, 128px/16 slices (~0.9 GB cache), a few epochs. Confirms the whole pipeline works against real Kaggle DICOM before spending a full session on more.

In [ ]:
def run_pipeline(config_path, submission_out):
    """preprocess -> train -> evaluate -> predict, entirely in this process.
    Returns the dict from run_training plus the eval/predict results, so
    every downstream step reads paths from the step before it -- never a
    hand-typed checkpoint path."""
    cfg = load_config(config_path)
    print(f"\n=== {cfg['run_name']} ===")

    preprocess_script.preprocess_split(cfg, 'train', research_root=REPO_ROOT)
    preprocess_script.preprocess_split(cfg, 'test', research_root=REPO_ROOT)

    train_result = train_script.run_training(cfg, REPO_ROOT, device=device)
    best_ckpt = train_result['checkpoint_dir'] / 'best.pt'
    print('Best checkpoint:', best_ckpt)
    print('Final val macro AUC:', train_result['final_val_metrics'].get('macro_auc'))

    eval_result = evaluate_script.evaluate_and_save(cfg, str(best_ckpt), 'val', device, research_root=REPO_ROOT)
    print('Val metrics written to:', eval_result['out_path'])

    submission_path = predict_script.run_prediction(
        cfg, str(best_ckpt), device, research_root=REPO_ROOT, out_path=submission_out,
    )
    print('Submission written to:', submission_path)
    return {'cfg': cfg, 'train_result': train_result, 'eval_result': eval_result, 'submission_path': submission_path}


smoke_result = run_pipeline(
    REPO_ROOT / 'configs' / 'kaggle_smoke.yaml',
    submission_out='/kaggle/working/submission_smoke.csv',
)

## Step 2 — Baseline run (2,000-study subset)

Only run this once Step 1 looks right. `configs/kaggle_baseline.yaml`: M1, 2,000 studies (~11 GB cache), 15 epochs with early stopping.

In [ ]:
baseline_result = run_pipeline(
    REPO_ROOT / 'configs' / 'kaggle_baseline.yaml',
    submission_out='/kaggle/working/submission.csv',
)

## Notes

- **Disk**: check `!df -h /kaggle/working` yourself before a long run -- the 19.5 GB quota is confirmed real, and every config above stays under a `data.max_studies` cap rather than assuming a number fits. `configs/kaggle_baseline_2500.yaml` is a bigger (2,500-study) capped option if you want more coverage without changing the caching strategy; see `README.md` for the other options (bounded LRU cache eviction, or no persistent cache at all) if/when you want to move past a capped subset.
- **Next steps in the M1->M7 progression**: `configs/m2_slice_attention.yaml` .. `configs/m6_report_weak.yaml` / `configs/final.yaml` exist but don't yet have Kaggle-path variants (they still point at local repo-relative `data/...`, useful for local dev only) -- ask for `kaggle_m2.yaml`..`kaggle_m6.yaml`/`kaggle_final.yaml`-style variants once `kaggle_baseline` has a real validated macro-AUC to compare against.
- **Before a real (graded) submission**, not just interactive development: check this competition's Code Requirements tab for whether internet access is disabled during the scored run. If so, this notebook (which needs internet for `git clone`/`pip install`) is for *training* only -- save the resulting checkpoint as a Kaggle Dataset, then use a second, minimal, internet-off notebook that loads it and calls `predict_script.run_prediction(...)` to produce the graded submission.